# 02 — Chronological forecasting experiments

This notebook follows `01_Data_Exploration.ipynb`, which remains unchanged. It describes the chronological 28-day experiment and summarizes the corrected 5,000-series pilot with price features and series-cluster bootstrap intervals.

**Research question:** How effectively does an AI-based model forecast retail demand under volatile demand conditions, compared with traditional baselines?

The experiment uses seed 42, three non-overlapping 28-day holdouts, and 1,000 bootstrap replicates. Volatility labels are recalculated from the preceding 52 complete weeks at each fold cutoff.

## Results from the 5,000-series pilot

Pooled WAPE combines absolute error and actual demand across all folds. Positive improvement means the candidate has lower WAPE than the reference. The confidence intervals resample series IDs and keep each selected series’ fold observations together.

| Model | Pooled WAPE | Pooled MAE |
|---|---:|---:|
| 7-day seasonal naive | 0.8753 | 1.2085 |
| Four-week weekday mean | 0.7764 | 1.0721 |
| LightGBM, no price | 0.6937 | 0.9578 |
| LightGBM, scheduled price features | 0.6845 | 0.9451 |

Against the stronger traditional baseline, price-aware LightGBM improved WAPE by **0.0919 points** overall (95% interval **0.0843–0.0998**). Adding scheduled price features to LightGBM improved WAPE by **0.0092 points** (95% interval **0.0080–0.0103**).

| Volatility group | Price-aware LightGBM WAPE | Weekday-mean WAPE | Improvement | 95% interval |
|---|---:|---:|---:|---:|
| Low | 0.5765 | 0.6206 | 0.0441 | 0.0374–0.0518 |
| Moderate | 0.8205 | 0.9549 | 0.1345 | 0.1228–0.1462 |
| High | 0.8451 | 1.0272 | 0.1821 | 0.1618–0.2032 |

These are pilot estimates for one seeded sample, not full-population or cross-retailer conclusions. Intervals are conditional on the sampled series and fixed model setup; the bootstrap-positive share is not a p-value. Future sell prices are treated as scheduled and known at forecast time.

## Load the report or a fresh run

To regenerate outputs, run `python src/forecast_experiments.py --max-series 5000 --bootstrap-replicates 1000` from the repository root first. The notebook uses fresh local outputs under `artifacts/` when present, otherwise the committed pilot snapshot under `reports/m5_5000_pilot/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "forecast_experiments.py").exists())
ARTIFACTS = ROOT / "artifacts"
if not (ARTIFACTS / "pooled_metrics.csv").exists():
    ARTIFACTS = ROOT / "reports" / "m5_5000_pilot"
manifest = json.loads((ARTIFACTS / "run_manifest.json").read_text(encoding="utf-8"))
fold_path = ARTIFACTS / "metrics.csv"
if not fold_path.exists(): fold_path = ARTIFACTS / "fold_metrics.csv"
fold_metrics = pd.read_csv(fold_path)
pooled_metrics = pd.read_csv(ARTIFACTS / "pooled_metrics.csv")
bootstrap = pd.read_csv(ARTIFACTS / "bootstrap_comparison.csv")
print({k: manifest[k] for k in ["seed", "series_evaluated", "horizon_days", "folds", "bootstrap_replicates", "price_features_used"]})

In [ ]:
overall = pooled_metrics[pooled_metrics["volatility_group"] == "all"]
display(overall[["model", "pooled_WAPE", "pooled_MAE", "mean_fold_WAPE", "mean_fold_MAE"]].sort_values("pooled_WAPE").round(4))
display(pooled_metrics.pivot(index="volatility_group", columns="model", values="pooled_WAPE").round(4))

In [ ]:
display(bootstrap[["volatility_group", "comparison", "reference_model", "candidate_model", "WAPE_improvement", "improvement_95CI_low", "improvement_95CI_high", "relative_improvement_pct", "bootstrap_probability_improvement_positive", "series"]].round(4))

## Interpretation and next step

The pilot supports a full-population M5 evaluation after checking available memory. Keep the folds and volatility definitions fixed, then test the price-feature ablation on all series. Since these results use a single retail dataset, describe them as M5 evidence and avoid generalizing directly to every retailer.